# 🏛️ Delhi Tourism Assistant
### Find nearby Hotels, Restaurants, Hospitals, Tourist Places + Live Weather
---

## 📦 Step 1: Import Libraries & Load Datasets

In [54]:
import pandas as pd
import math
import requests
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Load datasets
tourist_df    = pd.read_csv("Delhi_Tourist_Places_Final.csv")
hotels_df     = pd.read_csv("delhi_hotels_final_processed.csv")
hospital_df   = pd.read_csv("final_hospital.csv")
restaurants_df=pd.read_csv("zom.csv")
shopping_df = pd.read_csv("delhi_unique_shopping_markets_dataset.csv")


# Note: restaurants dataset (zom.csv) is optional — skip if not available
try:
    restaurants_df = pd.read_csv("zom.csv")
    RESTAURANTS_AVAILABLE = True
    print("✅ Restaurants dataset loaded")
except FileNotFoundError:
    RESTAURANTS_AVAILABLE = False
    print("⚠️  zom.csv not found — restaurant section will be skipped")

print(f"✅ Tourist Places : {tourist_df.shape[0]} records")
print(f"✅ Hotels         : {hotels_df.shape[0]} records")
print(f"✅ Hospitals      : {hospital_df.shape[0]} records")
print(f"✅ Shopping Markets : {shopping_df.shape[0]} records")

✅ Restaurants dataset loaded
✅ Tourist Places : 19 records
✅ Hotels         : 65 records
✅ Hospitals      : 320 records
✅ Shopping Markets : 49 records


In [47]:
import folium
from folium.plugins import MarkerCluster

def create_tourism_map(selected_place, nearest_hotels, nearest_hospitals, 
                        nearby_places, weather, nearest_restaurants=None):
    """
    Creates an interactive folium map with all recommendations
    color coded by category.
    """
    ref_lat = selected_place["Latitude"]
    ref_lon = selected_place["Longitude"]
    
    # ── Base Map ─────────────────────────────────────────────
    m = folium.Map(location=[ref_lat, ref_lon], zoom_start=13,
                   tiles="OpenStreetMap")
    
    # ── Weather Banner on Map ────────────────────────────────
    if weather:
        weather_html = f"""
        <div style="position:fixed; top:10px; right:10px; z-index:1000;
                    background:white; padding:10px; border-radius:8px;
                    box-shadow:2px 2px 6px rgba(0,0,0,0.3); font-family:Arial;">
            <b>🌤️ Live Weather</b><br>
            🌡️ {weather['temperature']}°C (Feels {weather['feels_like']}°C)<br>
            🌥️ {weather['description']}<br>
            💧 Humidity: {weather['humidity']}%<br>
            🌬️ Wind: {weather['wind_speed']} m/s<br>
            🌧️ Rain: {weather['rain_1h']} mm
        </div>
        """
        m.get_root().html.add_child(folium.Element(weather_html))

    # ── 📍 Selected Tourist Place (RED STAR) ─────────────────
    folium.Marker(
        location=[ref_lat, ref_lon],
        popup=folium.Popup(f"""
            <b>📍 {selected_place['Name'].strip()}</b><br>
            Type: {selected_place['Type']}<br>
            ⭐ Rating: {selected_place['Google review rating']}<br>
            💰 Entry: ₹{selected_place['Entrance Fee in INR']}<br>
            🕐 Visit: {selected_place['time needed to visit in hrs']} hrs<br>
            🌅 Best Time: {selected_place['Best Time to visit']}
        """, max_width=250),
        tooltip=f"📍 {selected_place['Name'].strip()} (Selected)",
        icon=folium.Icon(color="red", icon="star", prefix="fa")
    ).add_to(m)

    # ── 🏨 Hotels (BLUE) ────────────────────────────────────
    hotel_cluster = MarkerCluster(name="🏨 Hotels").add_to(m)
    for _, row in nearest_hotels.iterrows():
        folium.Marker(
            location=[row["Latitude"], row["Longitude"]],
            popup=folium.Popup(f"""
                <b>🏨 {row['Hotel Name']}</b><br>
                ⭐ Rating: {row['Rating']}<br>
                🌟 Stars: {row.get('Star Rating', 'N/A')}<br>
                💰 Price: ₹{int(row['Price']) if pd.notna(row['Price']) else 'N/A'}<br>
                📏 Distance: {row['distance_km']:.2f} km
            """, max_width=220),
            tooltip=f"🏨 {row['Hotel Name']}",
            icon=folium.Icon(color="blue", icon="bed", prefix="fa")
        ).add_to(hotel_cluster)

    # ── 🏥 Hospitals (GREEN) ────────────────────────────────
    hospital_cluster = MarkerCluster(name="🏥 Hospitals").add_to(m)
    for _, row in nearest_hospitals.iterrows():
        folium.Marker(
            location=[row["LATITUDE"], row["LONGITUDE"]],
            popup=folium.Popup(f"""
                <b>🏥 {row['Hospital Name']}</b><br>
                📍 {row['Address']}<br>
                📏 Distance: {row['distance_km']:.2f} km
            """, max_width=220),
            tooltip=f"🏥 {row['Hospital Name']}",
            icon=folium.Icon(color="green", icon="plus", prefix="fa")
        ).add_to(hospital_cluster)

    # ── 🌍 Nearby Tourist Places (ORANGE) ───────────────────
    tourist_cluster = MarkerCluster(name="🌍 Tourist Places").add_to(m)
    for _, row in nearby_places.iterrows():
        folium.Marker(
            location=[row["Latitude"], row["Longitude"]],
            popup=folium.Popup(f"""
                <b>🌍 {row['Name'].strip()}</b><br>
                Type: {row['Type']}<br>
                ⭐ Rating: {row['Google review rating']}<br>
                💰 Entry: ₹{row['Entrance Fee in INR']}<br>
                📏 Distance: {row['distance_km']:.2f} km
            """, max_width=220),
            tooltip=f"🌍 {row['Name'].strip()}",
            icon=folium.Icon(color="orange", icon="camera", prefix="fa")
        ).add_to(tourist_cluster)

    # ── 🍽️ Restaurants (PURPLE) ─────────────────────────────
    if nearest_restaurants is not None and RESTAURANTS_AVAILABLE:
        rest_cluster = MarkerCluster(name="🍽️ Restaurants").add_to(m)
        for _, row in nearest_restaurants.iterrows():
            folium.Marker(
                location=[row["Latitude"], row["Longitude"]],
                popup=folium.Popup(f"""
                    <b>🍽️ {row['Restaurant_Name']}</b><br>
                    ⭐ Rating: {round(row['Dining_Rating'], 2)}<br>
                    💰 For 2: ₹{row['Pricing_for_2']}<br>
                    📏 Distance: {row['distance_km']:.2f} km
                """, max_width=220),
                tooltip=f"🍽️ {row['Restaurant_Name']}",
                icon=folium.Icon(color="purple", icon="cutlery", prefix="fa")
            ).add_to(rest_cluster)

    # ── Legend ───────────────────────────────────────────────
    legend_html = """
    <div style="position:fixed; bottom:30px; left:30px; z-index:1000;
                background:white; padding:12px; border-radius:8px;
                box-shadow:2px 2px 6px rgba(0,0,0,0.3); font-family:Arial; font-size:13px;">
        <b>🗺️ Map Legend</b><br><br>
        🔴 Selected Tourist Place<br>
        🔵 Hotels<br>
        🟢 Hospitals<br>
        🟠 Nearby Tourist Places<br>
        🟣 Restaurants
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # ── Layer Control (toggle on/off) ────────────────────────
    folium.LayerControl(collapsed=False).add_to(m)

    return m

print("✅ Map function ready")

✅ Map function ready


## 🌤️ Step 2: Weather Setup (OpenWeatherMap API)
**Get your free API key at:** https://openweathermap.org/api  
Replace `your_api_key_here` below with your actual key.

In [48]:
# ============================================================
#  🔑 PASTE YOUR OPENWEATHERMAP API KEY HERE
# ============================================================
WEATHER_API_KEY = "d5817a4adf68f8fe8ada217ff2f7de4c"

def get_weather(lat, lon):
    """
    Fetches real-time weather for a given latitude/longitude.
    Returns a dict with temperature, humidity, wind, rain, description.
    """
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": WEATHER_API_KEY,
        "units": "metric"   # Celsius; change to 'imperial' for Fahrenheit
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if response.status_code != 200:
            print(f"⚠️  Weather API Error: {data.get('message', 'Unknown error')}")
            return None

        return {
            "temperature" : data["main"]["temp"],
            "feels_like"  : data["main"]["feels_like"],
            "humidity"    : data["main"]["humidity"],
            "wind_speed"  : data["wind"]["speed"],
            "rain_1h"     : data.get("rain", {}).get("1h", 0),
            "description" : data["weather"][0]["description"].capitalize()
        }
    except requests.exceptions.RequestException as e:
        print(f"⚠️  Could not connect to weather API: {e}")
        return None

def print_weather(place_name, weather):
    """Pretty-prints weather info."""
    if weather:
        print(f"\n🌤️  Live Weather at {place_name}")
        print("  " + "-"*38)
        print(f"  🌡️  Temperature  : {weather['temperature']}°C  (Feels like {weather['feels_like']}°C)")
        print(f"  🌥️  Condition    : {weather['description']}")
        print(f"  💧  Humidity     : {weather['humidity']}%")
        print(f"  🌬️  Wind Speed   : {weather['wind_speed']} m/s")
        print(f"  🌧️  Rain (1 hr)  : {weather['rain_1h']} mm")
        print("  " + "-"*38)
    else:
        print("  ⚠️  Weather data not available")

print("✅ Weather functions ready")

✅ Weather functions ready


In [49]:
# ============================================================
#  🌧️ WEATHER-BASED SMART RECOMMENDATIONS
# ============================================================

def get_weather_recommendation(weather, tourist_df, ref_lat, ref_lon):
    """
    If it's raining → recommends indoor places (Museums, Cafes, Temples etc.)
    If weather is clear → recommends all nearby places normally
    """
    if weather is None:
        return

    rain        = weather["rain_1h"]
    description = weather["description"].lower()
    is_raining  = rain > 0 or "rain" in description or "drizzle" in description or "thunderstorm" in description

    if is_raining:
        print("\n🌧️  It's raining! Here are some INDOOR places you can visit instead:\n")

        # Indoor-friendly types from your tourist dataset
        indoor_types = ["Museum", "Temple", "Monument", "Observatory",
                        "Art Gallery", "Science Centre", "Historical", "Religious"]

        # Calculate distance for all places
        tourist_df["distance_km"] = tourist_df.apply(
            lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
            axis=1
        )

        # Filter indoor places within 10 km
        indoor_places = tourist_df[
            tourist_df["Type"].str.contains("|".join(indoor_types), case=False, na=False)
        ].sort_values("distance_km")

        if not indoor_places.empty:
            print("🏛️  Nearby Indoor Tourist Spots:")
            print(indoor_places[["Name", "Type", "Google review rating",
                                  "Entrance Fee in INR", "distance_km"]]
                  .head(5)
                  .rename(columns={"distance_km": "Distance (km)",
                                   "Google review rating": "Rating"})
                  .to_string(index=False))
        else:
            print("  No indoor tourist spots found nearby.")

        # Recommend nearby hotels as shelter/cafe option
        hotels_df["distance_km"] = hotels_df.apply(
            lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
            axis=1
        )
        nearby_cafes = hotels_df.sort_values("distance_km").head(3)

        print("\n☕ Nearby Hotels/Cafes to take shelter:")
        print(nearby_cafes[["Hotel Name", "Rating", "Price", "distance_km"]]
              .rename(columns={"distance_km": "Distance (km)"})
              .to_string(index=False))

        print("\n💡 Tip: Carry an umbrella ☂️ or wait for the rain to stop before visiting outdoor spots!")

    else:
        print(f"\n✅ Weather looks good ({weather['description']}) — great time to explore outdoor places too!")

print("✅ Weather recommendation function ready")

✅ Weather recommendation function ready


## 📐 Step 3: Distance Function (Haversine Formula)

In [50]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculates the great-circle distance (km) between two GPS coordinates.
    """
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

print("✅ Haversine distance function ready")

✅ Haversine distance function ready


## 🤖 Step 4: Train KNN Recommendation Models

In [51]:
# ── Hotels KNN ──────────────────────────────────────────────
hotels_df["Price"] = (
    hotels_df["Price"]
    .astype(str).str.replace(",", "", regex=False)
)
hotels_df["Price"]                = pd.to_numeric(hotels_df["Price"],                errors="coerce")
hotels_df["Rating"]               = pd.to_numeric(hotels_df["Rating"],               errors="coerce")
hotels_df["Star Rating"]          = pd.to_numeric(hotels_df["Star Rating"],          errors="coerce")
hotels_df["Distance to Landmark"] = (
    hotels_df["Distance to Landmark"]
    .astype(str).str.replace(" km", "", regex=False)
)
hotels_df["Distance to Landmark"] = pd.to_numeric(hotels_df["Distance to Landmark"], errors="coerce")

hotel_features = ["Rating", "Price", "Star Rating", "Distance to Landmark"]
X_hotels       = hotels_df[hotel_features].fillna(hotels_df[hotel_features].mean())

scaler_hotels  = StandardScaler()
X_hotels_scaled = scaler_hotels.fit_transform(X_hotels)

knn_hotels = NearestNeighbors(n_neighbors=6, metric="euclidean")
knn_hotels.fit(X_hotels_scaled)
print("✅ Hotels KNN model trained")


# ── Restaurants KNN (if available) ──────────────────────────
if RESTAURANTS_AVAILABLE:
    rest_features  = ["Dining_Rating", "Pricing_for_2"]
    X_rest         = restaurants_df[rest_features].fillna(restaurants_df[rest_features].mean())
    scaler_rest    = StandardScaler()
    X_rest_scaled  = scaler_rest.fit_transform(X_rest)
    knn_rest       = NearestNeighbors(n_neighbors=6, metric="euclidean")
    knn_rest.fit(X_rest_scaled)
    print("✅ Restaurants KNN model trained")
else:
    print("⚠️  Restaurants KNN skipped (dataset not available)")

✅ Hotels KNN model trained
✅ Restaurants KNN model trained


## 🔍 Step 5: Recommendation Helper Functions

In [52]:
def recommend_similar_hotels(hotel_name, top_n=5):
    """Returns KNN-based similar hotels for a given hotel name."""
    matches = hotels_df[hotels_df["Hotel Name"] == hotel_name]
    if matches.empty:
        print(f"Hotel '{hotel_name}' not found.")
        return None
    idx           = matches.index[0]
    vec           = X_hotels_scaled[idx].reshape(1, -1)
    _, indices    = knn_hotels.kneighbors(vec)
    similar       = hotels_df.iloc[indices[0][1:top_n+1]]
    return similar[["Hotel Name", "Rating", "Price", "Star Rating"]]


def recommend_similar_restaurants(restaurant_name, top_n=5):
    """Returns KNN-based similar restaurants for a given restaurant name."""
    if not RESTAURANTS_AVAILABLE:
        print("⚠️  Restaurants dataset not loaded.")
        return None
    matches = restaurants_df[restaurants_df["Restaurant_Name"] == restaurant_name]
    if matches.empty:
        print(f"Restaurant '{restaurant_name}' not found.")
        return None
    idx           = matches.index[0]
    vec           = X_rest_scaled[idx].reshape(1, -1)
    _, indices    = knn_rest.kneighbors(vec)
    similar       = restaurants_df.iloc[indices[0][1:top_n+1]]
    return similar[["Restaurant_Name", "Dining_Rating", "Pricing_for_2"]]


print("✅ Recommendation functions ready")

✅ Recommendation functions ready


## 🗺️ Step 6: Main — Enter a Tourist Place
Run this cell and type any Delhi tourist place to get:
- 🌤️ Live weather at that location
- 🏨 Top 5 nearest hotels
- 🍽️ Top 5 nearest restaurants (if dataset available)
- 🏥 Top 5 nearest hospitals
- 🌍 Top 5 nearby tourist places

**Available places:** India Gate, Humayun's Tomb, Akshardham Temple, Waste to Wonder Park, Jantar Mantar, Chandni Chowk, Lotus Temple, Red Fort, Agrasen ki Baoli, Sunder Nursery, Garden of Five Senses, Lodhi Garden, National Gallery of Modern Art, National Zoological Park, Qutub Minar, National Science Centre, Gurudwara Bangla Sahib, Jama Masjid, Rail Museum

In [53]:
# ============================================================
#  🎯 USER INPUT
# ============================================================
place_name = input("Enter Tourist Place Name: ").strip()

# Case-insensitive match
tourist_df["Name_lower"] = tourist_df["Name"].str.strip().str.lower()

if place_name.lower() not in tourist_df["Name_lower"].values:
    print(f"\n❌ '{place_name}' not found. Please check the spelling.")
    print("Available places:", list(tourist_df["Name"].str.strip().values))
else:
    selected_place = tourist_df[tourist_df["Name_lower"] == place_name.lower()].iloc[0]
    ref_lat        = selected_place["Latitude"]
    ref_lon        = selected_place["Longitude"]

    # ── Header ──────────────────────────────────────────────
    print("\n" + "="*50)
    print(f"  📍 Selected Place : {selected_place['Name'].strip()}")
    print(f"  🏷️  Type           : {selected_place['Type']}")
    print(f"  ⭐ Google Rating   : {selected_place['Google review rating']}")
    print(f"  🕐 Visit Duration  : {selected_place['time needed to visit in hrs']} hrs")
    print(f"  💰 Entrance Fee    : ₹{selected_place['Entrance Fee in INR']}")
    print(f"  🌅 Best Time       : {selected_place['Best Time to visit']}")
    print(f"  📅 Weekly Off      : {selected_place['Weekly Off']}")
    print(f"  📷 DSLR Allowed    : {selected_place['DSLR Allowed']}")
    print("="*50)

    # ── 🌤️ LIVE WEATHER ─────────────────────────────────────
    weather = get_weather(ref_lat, ref_lon)
    print_weather(selected_place['Name'].strip(), weather)

    # ── 🏨 NEAREST HOTELS ───────────────────────────────────
    hotels_df["distance_km"] = hotels_df.apply(
        lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
        axis=1
    )
    nearest_hotels = hotels_df.sort_values("distance_km").head(5)

    print("\n🏨 Top 5 Nearest Hotels:")
    print(nearest_hotels[["Hotel Name", "Rating", "Star Rating", "Price", "distance_km"]]
          .rename(columns={"distance_km": "Distance (km)"})
          .to_string(index=False))

    # ── 🍽️ NEAREST RESTAURANTS ──────────────────────────────
    if RESTAURANTS_AVAILABLE:
        restaurants_df["distance_km"] = restaurants_df.apply(
            lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
            axis=1
        )
        nearest_restaurants = restaurants_df.sort_values("distance_km").head(5)
        print("\n🍽️  Top 5 Nearest Restaurants:")
        print(nearest_restaurants[["Restaurant_Name", "Dining_Rating", "Pricing_for_2", "distance_km"]]
              .rename(columns={"distance_km": "Distance (km)"})
              .to_string(index=False))
    else:
        print("\n🍽️  Restaurants: dataset not loaded (add zom.csv to enable)")

    # ── 🏥 NEAREST HOSPITALS ────────────────────────────────
    hospital_df["distance_km"] = hospital_df.apply(
        lambda row: haversine(ref_lat, ref_lon, row["LATITUDE"], row["LONGITUDE"]),
        axis=1
    )
    nearest_hospitals = hospital_df.sort_values("distance_km").head(5)

    print("\n🏥 Top 5 Nearest Hospitals:")
    print(nearest_hospitals[["Hospital Name", "Address", "distance_km"]]
          .rename(columns={"distance_km": "Distance (km)"})
          .to_string(index=False))

    # ── 🌍 NEARBY TOURIST PLACES ────────────────────────────
    tourist_df["distance_km"] = tourist_df.apply(
        lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
        axis=1
    )
    nearby_places = (
        tourist_df[tourist_df["distance_km"] > 0]
        .sort_values("distance_km")
        .head(5)
    )

    print("\n🌍 Top 5 Nearby Tourist Places:")
    print(nearby_places[["Name", "Type", "Google review rating", "distance_km"]]
          .rename(columns={"distance_km": "Distance (km)", "Google review rating": "Rating"})
          .to_string(index=False))

    print("\n" + "="*50)
    print("  ✅ Done! Happy exploring Delhi 🏛️")
    # ── 🗺️ GENERATE MAP ─────────────────────────────────────
    print("\n🗺️  Generating interactive map...")
    tourism_map = create_tourism_map(
        selected_place   = selected_place,
        nearest_hotels   = nearest_hotels,
        nearest_hospitals= nearest_hospitals,
        nearby_places    = nearby_places,
        weather          = weather,
        nearest_restaurants = nearest_restaurants if RESTAURANTS_AVAILABLE else None
    )
    tourism_map.save("delhi_tourism_map.html")
    print("✅ Map saved as 'delhi_tourism_map.html' — open it in your browser!")
    display(tourism_map)   # also shows inline inside Jupyter
    print("="*50)

Enter Tourist Place Name:  india gate



  📍 Selected Place : India Gate
  🏷️  Type           : War Memorial
  ⭐ Google Rating   : 4.6
  🕐 Visit Duration  : 0.5 hrs
  💰 Entrance Fee    : ₹0
  🌅 Best Time       : Evening
  📅 Weekly Off      : Not Available
  📷 DSLR Allowed    : Yes

🌤️  Live Weather at India Gate
  --------------------------------------
  🌡️  Temperature  : 26.21°C  (Feels like 26.21°C)
  🌥️  Condition    : Clear sky
  💧  Humidity     : 17%
  🌬️  Wind Speed   : 2.91 m/s
  🌧️  Rain (1 hr)  : 0 mm
  --------------------------------------

🏨 Top 5 Nearest Hotels:
                                               Hotel Name  Rating  Star Rating  Price  Distance (km)
                                                 The Hans     3.6          4.0   4860       1.524587
               The Connaught, New Delhi - IHCL SeleQtions     4.5          4.0   7499       2.168921
                                     Jukaso Inn Down Town     3.6          3.0   3134       2.168921
                                       The Park New D

## 🤝 Step 7 (Bonus): Get Similar Hotel Recommendations

In [11]:
# Change hotel name below to any hotel from the dataset
hotel_input = input("Enter Hotel Name for similar recommendations: ").strip()

result = recommend_similar_hotels(hotel_input)
if result is not None:
    print(f"\n🏨 Hotels similar to '{hotel_input}':")
    print(result.to_string(index=False))

Enter Hotel Name for similar recommendations:  Radisson Blu Plaza Delhi Airport



🏨 Hotels similar to 'Radisson Blu Plaza Delhi Airport':
                                  Hotel Name  Rating  Price  Star Rating
                        The Suryaa New Delhi     4.1   5348          5.0
                          Crowne Plaza Okhla     4.3   6300          5.0
                         Crowne Plaza Rohini     4.3   6498          5.0
           Holiday Inn New Delhi Mayur Vihar     4.1   6249          5.0
Welcomhotel by ITC Hotels, Dwarka, New Delhi     4.0   6223          5.0


## 🍽️ Step 8 (Bonus): Get Similar Restaurant Recommendations

In [12]:
# Only works if zom.csv is loaded
if RESTAURANTS_AVAILABLE:
    rest_input = input("Enter Restaurant Name for similar recommendations: ").strip()
    result = recommend_similar_restaurants(rest_input)
    if result is not None:
        print(f"\n🍽️  Restaurants similar to '{rest_input}':")
        print(result.to_string(index=False))
else:
    print("⚠️  Add zom.csv to your project folder to enable restaurant recommendations.")

Enter Restaurant Name for similar recommendations:  Gulati



🍽️  Restaurants similar to 'Gulati':
                                           Restaurant_Name  Dining_Rating  Pricing_for_2
                                                    Bo Tai       0.554192       1.508507
                      Downtown - Diners & Living Beer Cafe       0.554192       1.187391
                                                   Comorin       0.653155       1.508507
                                                  Pa Pa Ya       0.653155       1.508507
The Great Kabab Factory - Radisson Blu Plaza Delhi Airport       0.455229       1.508507


In [45]:
pip install folium


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [folium]
Note: you may need to restart the kernel to use updated packages.
